In [2]:
# 忽略警告訊息
import warnings
warnings.filterwarnings("ignore")

# 匯入必要的套件
import matplotlib.pyplot as plt

# 使用 fivethirtyeight 風格，讓圖表更美觀
# plt.style.use('fivethirtyeight')

# 設定字型，確保中文能正常顯示
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = [
    "Microsoft JhengHei",
    "Microsoft YaHei",
    "SimHei",
    "Arial Unicode MS",
    "DejaVu Sans"
]

# 設定數學字型，確保負號能正常顯示
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["mathtext.fontset"] = "dejavusans"

In [3]:
# ============================================================
# 1. 讀取資料與初步檢視
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 只在這裡讀取一次資料。
data = pd.read_csv("ex2.csv")

print("資料形狀：", data.shape)
display(data.head())

# info() 可以快速檢查每個欄位的資料型態、非空值數量，以及記憶體使用情況。
display(data.info())

# describe() 可以快速檢查每個數值欄位的平均數、標準差、最小值與最大值。
display(data.describe().round(2))

資料形狀： (506, 14)


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    int64  
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    int64  
 9   TAX      506 non-null    int64  
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
 13  MEDV     506 non-null    float64
dtypes: float64(11), int64(3)
memory usage: 55.5 KB


None

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
count,506.00,506.00,506.00,506.00,506.00,506.00,506.00,506.00,506.00,506.00,506.00,506.00,506.00,506.00
mean,3.61,11.36,11.14,0.07,0.55,6.28,68.57,3.80,9.55,408.24,18.46,356.67,12.65,22.53
std,8.60,23.32,6.86,0.25,0.12,0.70,28.15,2.11,8.71,168.54,2.16,91.29,7.14,9.20
min,0.01,0.00,0.46,0.00,0.38,3.56,2.90,1.13,1.00,187.00,12.60,0.32,1.73,5.00
25%,0.08,0.00,5.19,0.00,0.45,5.89,45.02,2.10,4.00,279.00,17.40,375.38,6.95,17.02
50%,0.26,0.00,9.69,0.00,0.54,6.21,77.50,3.21,5.00,330.00,19.05,391.44,11.36,21.20
75%,3.68,12.50,18.10,0.00,0.62,6.62,94.07,5.19,24.00,666.00,20.20,396.22,16.96,25.00
max,88.98,100.00,27.74,1.00,0.87,8.78,100.00,12.13,24.00,711.00,22.00,396.90,37.97,50.00


In [5]:
# ============================================================
# 1. 簡單線性迴歸：使用 RM 預測 MEDV
# ============================================================

work = data.copy()

# 只使用 RM 這個特徵來預測 MEDV。
X_simple = work[["RM"]]
y = work["MEDV"]

# 建立並訓練簡單線性迴歸模型。
simple_model = LinearRegression()
simple_model.fit(X_simple, y)

# 使用訓練好的模型對訓練資料進行預測。
simple_pred = simple_model.predict(X_simple)

# 計算評估指標：MSE、RMSE、MAE、R²。
# 先算 y, 一個是資料讀出來的 ,一個是差距(再重聽)

simple_mse = mean_squared_error(y, simple_pred)
simple_rmse = np.sqrt(simple_mse)
simple_mae = mean_absolute_error(y, simple_pred)
simple_r2 = r2_score(y, simple_pred)

print("截距：", round(simple_model.intercept_, 4))
print("RM 係數：", round(simple_model.coef_[0], 4))
print("MSE：", round(simple_mse, 4))
print("RMSE：", round(simple_rmse, 4))
print("MAE：", round(simple_mae, 4))
print("R²：", round(simple_r2, 4))
# R²越高,越能解釋

# 使用新資料進行預測，這裡我們使用兩筆 RM 值分別為 6.5 和 7.0 的資料。
newData = [
    [6.5],
    [7.0]
]

# 使用簡單線性迴歸模型預測新資料的 MEDV。
new_pred = simple_model.predict(newData)
print("新資料預測結果：", new_pred.round(2))

截距： -34.6706
RM 係數： 9.1021
MSE： 43.6006
RMSE： 6.6031
MAE： 4.4478
R²： 0.4835
新資料預測結果： [24.49 29.04]
